# Entity Resolution using Hybrid Search Engine

**Goal:** Take the raw entities extracted by our NER models (Notebook 3) and query them against the canonical `master_df` using the existing `advanced_search_dynamic` engine (Notebook 4). Non-characters are filtered out based on the engine's `final_score`.

In [1]:
import pandas as pd
import numpy as np
import re
import ast
import itertools

import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

In [2]:
# =========================================================
# 1. SETUP & CONFIGURATION
# =========================================================
# Load your data
raw_ner_df = pd.read_csv('./data/GOT-characters-phrases-bert.csv')
master_df = pd.read_pickle('./data/got_search_engine_df.pkl')

In [3]:
# =========================================================
# 1. DATA PREPARATION & CLEANING
# =========================================================
# Ensure core text columns are safe
master_df['text'] = master_df['text'].fillna('')
master_df['title'] = master_df['title'].fillna('')


# =========================================================
# 2. GLOBAL SEARCH UTILITIES
# =========================================================
# Initialize and fit TF-IDF vectorizer once
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(master_df['text'])

def get_jaccard_sim(str1, str2): 
    a = set(re.findall(r'\w+', str1.lower())) 
    b = set(re.findall(r'\w+', str2.lower()))
    if not a or not b: return 0.0
    c = a.intersection(b)
    return float(len(c)) / (len(a) + len(b) - len(c))

ROLE_BOOSTS = {'pov': 2.0, 'appears': 1.0, 'mentioned': 0.2, 'other': 0.1, 'appendix': 0.0}

# =========================================================
# 3. THE DYNAMIC SEARCH ENGINE
# =========================================================
def advanced_search_dynamic(query_name, df, current_book=None, retrieve_by='tfidf', rank_by='hybrid', top_n=5):
    """
    Dynamic 3-Stage Search Pipeline: Context -> Retrieval -> Custom Ranking
    """
    query_lower = query_name.lower()
    
    # Pre-calculate TF-IDF for all rows safely
    query_vec = tfidf_vectorizer.transform([query_lower])
    tfidf_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    
    working_df = df.copy()
    working_df['tfidf_base_score'] = tfidf_scores
    
    # ---------------------------------------------------------
    # STAGE 0: CONTEXTUAL FILTERING (Hard Book Filter)
    # ---------------------------------------------------------
    if current_book:
        book_mask = working_df['books_parsed'].apply(lambda x: current_book in x)
        working_df = working_df[book_mask].copy()
        if working_df.empty:
            print(f"No candidates found in book: '{current_book}'")
            return pd.DataFrame(columns=['title', 'final_score'])

    # ---------------------------------------------------------
    # STAGE 1: RETRIEVAL (Finding Candidates)
    # ---------------------------------------------------------
    if retrieve_by == 'exact':
        mask = working_df['text'].str.lower().str.contains(query_lower, regex=False)
        retrieved_df = working_df[mask].copy()
    elif retrieve_by == 'jaccard':
        working_df['jaccard_base_score'] = working_df['text'].apply(lambda x: get_jaccard_sim(query_lower, x))
        retrieved_df = working_df[working_df['jaccard_base_score'] > 0].copy()
    elif retrieve_by == 'tfidf':
        retrieved_df = working_df[working_df['tfidf_base_score'] > 0].copy()
    else:
        raise ValueError("retrieve_by must be 'exact', 'jaccard', or 'tfidf'")
        
    if retrieved_df.empty:
        print("No candidates matched the retrieval criteria.")
        return pd.DataFrame(columns=['title', 'final_score'])

    # ---------------------------------------------------------
    # STAGE 2: DYNAMIC RANKING (Sorting Candidates)
    # ---------------------------------------------------------
    if rank_by == 'none':
        retrieved_df['final_score'] = 1.0 
        
    elif rank_by == 'jaccard':
        if 'jaccard_base_score' not in retrieved_df.columns:
            retrieved_df['final_score'] = retrieved_df['text'].apply(lambda x: get_jaccard_sim(query_lower, x))
        else:
            retrieved_df['final_score'] = retrieved_df['jaccard_base_score']
            
    elif rank_by == 'tfidf':
        retrieved_df['final_score'] = retrieved_df['tfidf_base_score']
        
    elif rank_by == 'hybrid':
        scores = []
        for idx, row in retrieved_df.iterrows():
            score = row['tfidf_base_score']
            
            # Metadata Boosts (Titles and Aliases)
            if query_lower in str(row['title']).lower(): score += 0.5
            aliases = row.get('aliases', [])
            if isinstance(aliases, str) and aliases.startswith('['):
                try: aliases = ast.literal_eval(aliases)
                except: aliases = []
            if isinstance(aliases, list) and any(query_lower in str(a).lower() for a in aliases):
                score += 0.3
            elif isinstance(aliases, str) and query_lower in aliases.lower():
                score += 0.3
                
            # Network Connectivity Boost
            score += (row.get('search_pagerank', 0.0) * 10)
            score += row.get('search_closeness', 0.0)
            
            # Contextual Role Boost (If reading a specific book)
            if current_book:
                character_role = row['books_parsed'].get(current_book, 'appendix')
                score += ROLE_BOOSTS.get(character_role, 0.0)
                
            scores.append(score)
        retrieved_df['final_score'] = scores
        
    # --- THE DYNAMIC COLUMN CHECK ---
    elif rank_by in retrieved_df.columns:
        retrieved_df['final_score'] = pd.to_numeric(retrieved_df[rank_by], errors='coerce').fillna(0.0)
    else:
        raise ValueError(f"rank_by '{rank_by}' is not recognized or not a column in the DataFrame.")
        
    # ---------------------------------------------------------
    # STAGE 3: SMART SORTING
    # ---------------------------------------------------------
    # If the ranking metric has 'distance' in its name, smaller is better (Ascending)
    # Otherwise, higher score is better (Descending)
    is_distance_metric = 'distance' in rank_by.lower()
    
    sorted_results = retrieved_df.sort_values(
        by='final_score', 
        ascending=is_distance_metric 
    )
    
    if current_book:
        # Extract the specific role they had in the current book to display it
        sorted_results['context_role'] = sorted_results['books_parsed'].apply(
            lambda x: x.get(current_book, 'appendix')
        )
        # Return the title, the score, and their role in the book
        return sorted_results[['title', 'final_score', 'context_role']].head(top_n)
    else:
        # Standard return if no book context was provided
        return sorted_results[['title', 'final_score']].head(top_n)

# =========================================================

In [4]:
# =========================================================
# 1. SETUP & CONFIGURATION
# =========================================================
BOOK_MAP = {
    1: "A Game of Thrones",
    2: "A Clash of Kings",
    3: "A Storm of Swords",
    4: "A Feast for Crows",
    5: "A Dance with Dragons"
}

# You may need to tweak this threshold now that queries are longer
MINIMUM_SCORE_THRESHOLD = 0.2

# Cache dictionary now includes the phrase: {(book_id, raw_name, phrase): "Canonical Name"}
resolution_cache = {}

# =========================================================
# 2. THE RESOLUTION FUNCTION (CONTEXT-AWARE)
# =========================================================
def get_canonical_name(raw_name, phrase, book_id, df):
    """Passes a combined string of NER character and context phrase to the search engine."""
    raw_name = str(raw_name).strip()
    
    # Use empty string if phrase is missing (e.g., when resolving Chapter names)
    phrase_text = str(phrase).strip() if pd.notna(phrase) else ""
    
    # The cache key MUST include the phrase so different contexts of "Jon" are evaluated separately
    cache_key = (book_id, raw_name, phrase_text)
    
    if cache_key in resolution_cache:
        return resolution_cache[cache_key]
        
    book_title = BOOK_MAP.get(book_id)
    
    # Combine the name and the phrase to give the TF-IDF vectorizer maximum context
    # If there is no phrase, just use the name
    query_string = f"{raw_name} {phrase_text}".strip()
    
    # Run the custom search engine
    results = advanced_search_dynamic(
        query_name=query_string, 
        df=df, 
        current_book=book_title, 
        retrieve_by='tfidf', 
        rank_by='hybrid', 
        top_n=1
    )
    
    # Check if we got a valid result above our threshold
    if not results.empty:
        top_score = results.iloc[0]['final_score']
        if top_score >= MINIMUM_SCORE_THRESHOLD:
            canonical_name = results.iloc[0]['title']
            resolution_cache[cache_key] = canonical_name
            return canonical_name
            
    # If no results or score is too low
    resolution_cache[cache_key] = None
    return None


In [5]:
# =========================================================
# 3. RUNNING THE PIPELINE (NO POV, JUST TARGETS)
# =========================================================
print(f"Starting with {len(raw_ner_df)} total NER extractions.")

unique_mentions = raw_ner_df[['book', 'chapter', 'character', 'phrase']].drop_duplicates().copy()

print("Resolving raw NER characters to Canonical Names using context...")
tqdm.pandas(desc="Processing Characters")
unique_mentions['canonical_name'] = unique_mentions.progress_apply(
    lambda row: get_canonical_name(row['character'], row['phrase'], row['book'], master_df), 
    axis=1
)

clean_targets = unique_mentions.dropna(subset=['canonical_name']).copy()

print("Removing Prologues and Epilogues...")
clean_targets = clean_targets[~clean_targets['chapter'].str.contains('Prologue|Epilogue', case=False)].copy()

Starting with 83174 total NER extractions.
Resolving raw NER characters to Canonical Names using context...


Processing Characters:   0%|          | 220/83174 [00:05<32:14, 42.89it/s]

No candidates matched the retrieval criteria.


Processing Characters:   2%|▏         | 1855/83174 [00:46<32:39, 41.49it/s]

No candidates matched the retrieval criteria.


...

No candidates matched the retrieval criteria.


Processing Characters:  98%|█████████▊| 81530/83174 [1:06:05<00:49, 32.89it/s]

No candidates matched the retrieval criteria.


Processing Characters: 100%|██████████| 83174/83174 [1:07:07<00:00, 20.65it/s]

Removing Prologues and Epilogues...


In [6]:
# =========================================================
# 4. GENERATING THE RELATIONSHIP MAP (CO-OCCURRENCE)
# =========================================================
print("\nGenerating Co-occurrence Edges...")

# Group characters by the specific book and chapter they appear in
chapter_groups = clean_targets.groupby(['book', 'chapter'])['canonical_name'].unique()

all_edges = []

# Iterate through every chapter's character list
for characters in chapter_groups:
    sorted_chars = sorted(characters)
    
    # Generate every unique pair of characters in this chapter
    pairs = list(itertools.combinations(sorted_chars, 2))
    all_edges.extend(pairs)

# Convert pairs into a DataFrame
edges_df = pd.DataFrame(all_edges, columns=['Source', 'Target'])

# Count occurrences to get Weight
relation_map = edges_df.groupby(['Source', 'Target']).size().reset_index(name='weight')
relation_map = relation_map.sort_values(by='weight', ascending=False)

print("\n--- Top Co-occurrence Relationships Found ---")
print(relation_map.head(10))

print(f"\nTotal Unique Characters (Nodes): {len(set(relation_map['Source']).union(set(relation_map['Target'])))}")
print(f"Total Relationships (Edges): {len(relation_map)}")

relation_map.to_csv('./data/got_character_edges_cooccurrence.csv', index=False)


Generating Co-occurrence Edges...

--- Top Co-occurrence Relationships Found ---
                  Source            Target  weight
1170  Daenerys Targaryen  Tyrion Lannister      52
947     Cersei Lannister   Jaime Lannister      48
1837            Jon Snow  Tyrion Lannister      44
1668     Jaime Lannister  Tyrion Lannister      41
1106  Daenerys Targaryen          Jon Snow      38
408           Arya Stark  Tyrion Lannister      38
332           Arya Stark          Jon Snow      36
1300      Davos Seaworth  Tyrion Lannister      36
1097  Daenerys Targaryen   Jaime Lannister      35
724           Bran Stark  Tyrion Lannister      33

Total Unique Characters (Nodes): 182
Total Relationships (Edges): 2358


In [8]:
import networkx as nx

# =========================================================
# 5. BUILDING THE GEPHI NETWORK (GEXF EXPORT)
# =========================================================
print("\nBuilding Network Graph for Gephi...")

# Initialize an Undirected Graph (co-occurrence in a chapter is a two-way relationship)
G = nx.Graph()

# Iterate through our relation_map and add edges with their weights
for idx, row in relation_map.iterrows():
    G.add_edge(
        row['Source'], 
        row['Target'], 
        weight=row['weight']
    )

print(f"Graph created with {G.number_of_nodes()} characters (nodes) and {G.number_of_edges()} relationships (edges).")

# --- Pre-calculate metrics to make Gephi styling easier ---
print("Calculating network metrics...")

# 1. Weighted Degree (Total number of mentions a character shares with others)
degree_dict = dict(G.degree(weight='weight'))
nx.set_node_attributes(G, degree_dict, 'weighted_degree')

# 2. PageRank (Measures the 'influence' of a character in the overall network)
pagerank_dict = nx.pagerank(G, weight='weight')
nx.set_node_attributes(G, pagerank_dict, 'pagerank')

# 3. Betweenness Centrality (Finds the 'bridge' characters between different storylines)
# Note: We use 1/weight for the distance, as higher weight means they are "closer"
distance_dict = {(u, v): 1 / d['weight'] for u, v, d in G.edges(data=True)}
nx.set_edge_attributes(G, distance_dict, 'distance')
betweenness_dict = nx.betweenness_centrality(G, weight='distance')
nx.set_node_attributes(G, betweenness_dict, 'betweenness')

# Export to GEXF format
output_file = './data/got_character_network.gexf'
nx.write_gexf(G, output_file)

print(f"\nSuccess! Network exported to: {output_file}")
print("You can now open this file directly in Gephi.")


Building Network Graph for Gephi...
Graph created with 182 characters (nodes) and 2358 relationships (edges).
Calculating network metrics...

Success! Network exported to: ./data/got_character_network.gexf
You can now open this file directly in Gephi.
